### Setup

In [93]:
import sys
import os
sys.path.append("../")
sys.path.append("../..")

In [94]:
import math

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [95]:
from scale_rl.common.wandb_utils import *

In [96]:
from color import METHOD_COLORS, METHOD_BOUNDARY_COLORS, BORDER_GRAY

In [97]:
from scale_rl.envs.humanoid_bench import HB_LOCOMOTION_NOHAND, HB_SUCCESS_SCORE, HB_RANDOM_SCORE

def replace_underbar_to_hypen(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list

def replace_underbar_to_hypen_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

In [98]:
HB_LOCOMOTION_NOHAND = replace_underbar_to_hypen(HB_LOCOMOTION_NOHAND)
HB_SUCCESS_SCORE = replace_underbar_to_hypen_dict(HB_SUCCESS_SCORE)
HB_RANDOM_SCORE = replace_underbar_to_hypen_dict(HB_RANDOM_SCORE)

# Remove manipulation & with-hand tasks
HB_SUCCESS_SCORE = {k: v for k, v in HB_SUCCESS_SCORE.items() if k in HB_LOCOMOTION_NOHAND}
HB_RANDOM_SCORE = {k: v for k, v in HB_RANDOM_SCORE.items() if k in HB_LOCOMOTION_NOHAND}

HB_STEPS = 1000000 # 1M

In [99]:
def normalize_score_with_random_and_base_score(
    df: pd.DataFrame, random_score_dict: dict, base_score_dict: dict
) -> pd.DataFrame:
    """
    Normalize the 'value' column based on random and base scores.
        normalized_value = (value - random_score) / (base_score - random_score)

    Args:
    - df (pd.DataFrame): DataFrame with 'env_name' and 'value' columns.
    - random_score_dict (dict): Mapping of 'env_name' to random scores.
    - base_score_dict (dict): Mapping of 'env_name' to base scores.

    Returns:
    - pd.DataFrame: DataFrame with an added 'normalized_value' column.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    df_normalized = df.copy()

    # Define a function to normalize a single value
    def normalize_value(row):
        env_name = row["env_name"]
        value = row["value"]
        base_score = base_score_dict[env_name]
        random_score = random_score_dict[env_name]

        return (value - random_score) / (base_score - random_score)

    # Apply the normalization function to each row
    df_normalized["value"] = df_normalized.apply(normalize_value, axis=1)

    return df_normalized

### Load Simba+ & SimbaV2

In [100]:
entity = 'draftrec'
project_name = 'Simba_2501'
run_exp_names_to_analysis_exp_names = {
    'hypersimba_hb_metrics': 'SimbaV2',
    'simba_hb_metrics': 'Simba',
}
run_exp_names = list(run_exp_names_to_analysis_exp_names.keys())
metrics = ['avg_return', 'avg_estimate_q', 'avg_discounted_return', 'critic_encoder/effective_pnorm_total', 'critic_encoder/effective_gnorm_total', 'critic_encoder/effective_lr_total',  'critic_encoder/effective_fnorm_total', 'critic_predictor/effective_pnorm_total', 'critic_predictor/effective_gnorm_total', 'critic_predictor/effective_lr_total',  'critic_predictor/effective_fnorm_total', 'critic/effective_pnorm_total', 'critic/effective_gnorm_total', 'critic/effective_lr_total',  'critic/effective_fnorm_total']

In [101]:
# Load from wandb
runs = collect_runs(entity=entity, project_name=project_name) 
filtered_runs = filter_runs(runs, exp_names = run_exp_names)
wandb_df = convert_runs_to_dataframe(
    runs = filtered_runs, 
    run_exp_name_to_analysis_exp_name=run_exp_names_to_analysis_exp_names
)
wandb_df = wandb_df[wandb_df.apply(lambda row: 'finished' in str(row['run']), axis=1)]

  0%|          | 0/2093 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
eval_df = convert_wandb_df_to_eval_df(wandb_df, metrics)
eval_df

  0%|          | 0/50 [00:00<?, ?it/s]

,exp_name,env_name,seed,metric,env_step,value
0,Simba,h1-stair-v0,4000,avg_return,0.0,3.689303
1,Simba,h1-stair-v0,4000,avg_return,100000.0,42.759250
2,Simba,h1-stair-v0,4000,avg_return,200000.0,198.748031
3,Simba,h1-stair-v0,4000,avg_return,300000.0,271.376898
4,Simba,h1-stair-v0,4000,avg_return,400000.0,334.306104
...,...,...,...,...,...,...
7695,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,600000.0,1.000000
7696,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,700000.0,1.000000
7697,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,800000.0,1.000000
7698,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,900000.0,1.000000


In [11]:
eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')
eval_df = eval_df[eval_df["env_name"].isin(HB_LOCOMOTION_NOHAND)]
eval_df

,exp_name,env_name,seed,metric,env_step,value
0,Simba,h1-stair-v0,4000,avg_return,0.0,3.689303
1,Simba,h1-stair-v0,4000,avg_return,100000.0,42.759250
2,Simba,h1-stair-v0,4000,avg_return,200000.0,198.748031
3,Simba,h1-stair-v0,4000,avg_return,300000.0,271.376898
4,Simba,h1-stair-v0,4000,avg_return,400000.0,334.306104
...,...,...,...,...,...,...
7695,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,600000.0,1.000000
7696,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,700000.0,1.000000
7697,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,800000.0,1.000000
7698,SimbaV2,h1-walk-v0,0,critic/effective_fnorm_total,900000.0,1.000000


In [12]:
eval_df[eval_df["metric"] == "avg_return"] = normalize_score_with_random_and_base_score(eval_df[eval_df["metric"] == "avg_return"], HB_RANDOM_SCORE, HB_SUCCESS_SCORE)
eval_df[eval_df["metric"] == "avg_estimate_q"] = normalize_score_with_random_and_base_score(eval_df[eval_df["metric"] == "avg_estimate_q"], HB_RANDOM_SCORE, HB_SUCCESS_SCORE)
eval_df[eval_df["metric"] == "avg_discounted_return"] = normalize_score_with_random_and_base_score(eval_df[eval_df["metric"] == "avg_discounted_return"], HB_RANDOM_SCORE, HB_SUCCESS_SCORE)

In [13]:
eval_df[eval_df["metric"] == "avg_return"]["value"]

0       0.000828
1       0.056892
2       0.280728
3       0.384947
4       0.475247
          ...   
7552    0.637703
7553    1.100155
7554    1.202632
7555    1.200120
7556    1.195048
Name: value, Length: 550, dtype: float64

In [14]:
eval_df["exp_name"].unique()

array(['Simba', 'SimbaV2'], dtype=object)

In [15]:
eval_df["metric"].unique()

array(['avg_return', 'avg_estimate_q', 'avg_discounted_return',
       'critic_encoder/effective_pnorm_total',
       'critic_encoder/effective_gnorm_total',
       'critic_encoder/effective_lr_total',
       'critic_encoder/effective_fnorm_total',
       'critic_predictor/effective_pnorm_total',
       'critic_predictor/effective_gnorm_total',
       'critic_predictor/effective_lr_total',
       'critic/effective_pnorm_total', 'critic/effective_gnorm_total',
       'critic/effective_lr_total', 'critic/effective_fnorm_total'],
      dtype=object)

In [16]:
simbav2_marker = 'o'
simba_marker = 'P'
simba_plus_marker = '^'

markers = {
    'SimbaV2': simbav2_marker,
    'SimbaV2_encoder': simbav2_marker,
    'SimbaV2_predictor': simbav2_marker,
    # 'Simba+': simba_plus_marker,
    'Simba': simba_marker,
    'Simba_encoder': simba_marker,
    'Simba_predictor': simba_marker,
}

In [80]:
simbav2_color = METHOD_BOUNDARY_COLORS["SimbaV2"] # sns.color_palette("YlOrBr",20)[8]
simbav2_alpha_color = METHOD_COLORS["SimbaV2"] # (*simba_color, 0.2)
simba_color = "#feb600" #  METHOD_BOUNDARY_COLORS["Simba"] # sns.color_palette("YlOrBr",20)[8]
simba_alpha_color = "#feb600" # METHOD_COLORS["Simba"] # (*simba_color, 0.2)

simbav2_encoder_color = "#8cb0f8"
simbav2_encoder_alpha_color = "#8cb0f8"
simbav2_predictor_color = "#5978ea"
simbav2_predictor_alpha_color = "#5978ea"

simba_encoder_color = "#ffcb10"
simba_encoder_alpha_color = "#ffcb10"
simba_predictor_color = "#e8a700"
simba_predictor_alpha_color = "#bd8800"
simba_predictor_tick_color = "#bd8800"

BORDER_GRAY = "#D3D3D3"
colors = {
    'SimbaV2': simbav2_color,
    'SimbaV2_encoder': simbav2_encoder_color,
    'SimbaV2_predictor': simbav2_predictor_color,
    # 'Simba+': simba_plus_color,
    'Simba': simba_color, 
    'Simba_encoder': simba_encoder_color,
    'Simba_predictor': simba_predictor_color,
    'Simba_predictor_tick': simba_predictor_tick_color,
}

In [81]:
import matplotlib.lines as mlines
simbav2_legend = mlines.Line2D([], [], color=simbav2_alpha_color, markeredgecolor=simbav2_color, marker=simbav2_marker, linestyle='None', markersize=14, label='SimbaV2')
simbav2_encoder_legend = mlines.Line2D([], [], color=simbav2_encoder_alpha_color, markeredgecolor=simbav2_encoder_color, marker=simbav2_marker, linestyle='None', markersize=14, label='SimbaV2 Encoder')
simbav2_predictor_legend = mlines.Line2D([], [], color=simbav2_predictor_alpha_color, markeredgecolor=simbav2_predictor_color, marker=simbav2_marker, linestyle='None', markersize=14, label='SimbaV2 Predictor')

simba_legend = mlines.Line2D([], [], color=simba_alpha_color, markeredgecolor=simba_color, marker=simba_marker, linestyle='None', markersize=14, label='Simba')
simba_encoder_legend = mlines.Line2D([], [], color=simba_encoder_alpha_color, markeredgecolor=simba_encoder_color, marker=simba_marker, linestyle='None', markersize=14, label='Simba Encoder')
simba_predictor_legend = mlines.Line2D([], [], color=simba_predictor_alpha_color, markeredgecolor=simba_predictor_color, marker=simba_marker, linestyle='None', markersize=14, label='Simba Predictor')

legend_handles = {
    'SimbaV2': simbav2_legend,
    'SimbaV2 Encoder': simbav2_encoder_legend,
    'SimbaV2 Predictor': simbav2_predictor_legend,
    'Simba': simba_legend, 
    'Simba Encoder': simba_encoder_legend,
    'Simba Predictor': simba_predictor_legend,
    # 'Simba+': simba_plus_legend, 
}

# For plot
marker_sizes_all = {
    'SimbaV2': 125,
    'SimbaV2_encoder': 125,
    'SimbaV2_predictor': 125,
    # 'Simba+': 150,
    'Simba': 150,
    'Simba_encoder': 150,
    'Simba_predictor': 150,
}
marker_sizes = {
    'SimbaV2': 125,
    'SimbaV2_encoder': 125,
    'SimbaV2_predictor': 125,
    # 'Simba+': 150,
    'Simba': 150,
    'Simba_encoder': 150,
    'Simba_predictor': 150,
}

### Visualization

### Total

In [82]:
TITLE_FONTSIZE = 18
TICK_FONTSIZE = 16
LABEL_FONTSIZE = 18

In [83]:
def simple_axis(ax):
    # Hide all spines for a simpler look
    for spine in ax.spines.values():
        spine.set_visible(False)

In [84]:
def subplot_q_bias_total(
    ax,
    experiments,
    metric_data,
    colors,
    num_x_ticks: int = 6,
    num_y_ticks: int = 5,
    x_label: str = "Env steps (M)",
    line_width: float = 2.5,
    y_lim_min: float = -1.0,
    y_lim_max: float = 1.0,
):  
    pivot_df = metric_data.pivot(index=["exp_name", "seed", "env_step", "env_name"], columns="metric", values="value").reset_index()
    # Calculate q_bias as the difference
    pivot_df["q_bias"] = (pivot_df["avg_estimate_q"] - pivot_df["avg_discounted_return"]) / pivot_df["avg_discounted_return"]

    # Melt back to long format, adding the new "q_bias" metric
    _metric_data = pd.melt(
        pivot_df,
        id_vars=["exp_name", "seed", "env_step", "env_name"],
        value_vars=["avg_discounted_return", "avg_estimate_q", "q_bias"],
        var_name="metric",
        value_name="value"
    )

    # Sort the result for consistency
    _metric_data = _metric_data[_metric_data["metric"] == "q_bias"]
    for exp in experiments:
        exp_data = _metric_data[_metric_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        
        # Check if env_step = 0 exists, if not, add it with value 0
        if 0 not in exp_data["env_step"].values:
            zero_row = pd.DataFrame({
                "env_step": [0],
                "value": [0],
                "exp_name": [exp],
                "metric": ["q_bias"]
            })
            exp_data = pd.concat([zero_row, exp_data]).reset_index(drop=True)
        
        # Group by env_step and calculate mean across all environments
        grouped_data = exp_data.groupby(["env_step"])["value"]
        env_steps = grouped_data.mean().index.values
        mean = grouped_data.mean().values
        std_error = grouped_data.sem().values  # Standard error of the mean

        # Plot mean history with thicker lines
        if exp == 'SimbaV2':
            zorder = 10
        elif exp == 'Simba':
            zorder = 0
        else:
            zorder = -10
    
            
        ax.plot(env_steps, 
                mean, 
                linewidth=line_width, 
                color=colors[exp], 
                zorder=zorder)
        # Draw markers
        timestep_list = [0, 2e5, 4e5, 6e5, 8e5, 1e6]
        markers_idx = list(range(0, len(mean), len(mean) // (num_x_ticks-1)))
        timestep_list = [t for t in timestep_list if t <= max(env_steps)]
        markers_idx = markers_idx[:len(timestep_list)]
        markers_values = np.interp(timestep_list, env_steps, mean)
        ax.scatter(timestep_list,
                    markers_values,
                    label=None,
                    color=colors[exp], 
                    edgecolor='white',
                    marker=markers[exp], 
                    s=marker_sizes[exp],
                    zorder=zorder+5)
        ax.fill_between(
            env_steps,
            mean - 1.96 * std_error,
            mean + 1.96 * std_error,
            alpha=0.2,
            color=colors[exp],
        )
            
    
    # Set labels and limits
    ax.set_xlabel(x_label, fontsize=TICK_FONTSIZE)

    # Set x-ticks and format them
    x_ticks = np.linspace(0, 1e6, num_x_ticks)
    ax.set_xticks(x_ticks)
    # Format x-tick labels
    x_tick_labels = []
    for tick in x_ticks:
        if tick == 0:
            x_tick_labels.append("0")
        else:
            x_tick_labels.append("{:.1f}".format(tick / 1e6))
    
    ax.set_xticklabels(x_tick_labels, fontsize=TICK_FONTSIZE)
    
    ax.set_autoscale_on(False)
    
    ax.xaxis.set_major_locator(plt.MaxNLocator(num_x_ticks, integer=True))
    ax.yaxis.set_major_locator(plt.MaxNLocator(num_y_ticks, integer=True))
    ax.xaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
    ax.yaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
    simple_axis(ax)
    
    # Set y-ticks
    y_ticks = np.linspace(y_lim_min, y_lim_max, num_y_ticks)
    ax.set_yticks(y_ticks)
    y_len = y_lim_max - y_lim_min
    ax.set_ylim(y_lim_min - 0.1*y_len, y_lim_max + 0.1*y_len)
    ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)
    
    from matplotlib.ticker import FuncFormatter
    def round_formatter(y, _):
        if y.is_integer():  # Check if the tick is an integer
            return f"{int(y)}"  # Convert to int for cleaner display
        return f"{y:.1f}"  # Keep one decimal place for floats
    ax.yaxis.set_major_formatter(FuncFormatter(round_formatter))
        
    # Add custom title with first line bold
    title = "Norm Q Bias"
    ax.set_title('')  # Clear any existing title
    ax.text(0.5, 1.08, title, color='black', transform=ax.transAxes, ha='center', va='bottom', fontsize=TITLE_FONTSIZE)


In [85]:
def subplot_q_variance_total(
    ax,
    experiments,
    metric_data,
    colors,
    num_x_ticks: int = 6,
    num_y_ticks: int = 5,
    x_label: str = "Env steps (M)",
    line_width: float = 2.5,
    y_lim_min: float = 0.0,
    y_lim_max: float = 4e-2,
):  
    _metric_data = metric_data[metric_data["metric"] == "avg_estimate_q"]
    for exp in experiments:
        exp_data = _metric_data[_metric_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        
        # Group by env_step and calculate mean across all environments
        std_per_env = exp_data.groupby(["env_name", "env_step"])["value"].std().reset_index(name="std")
        grouped_data = std_per_env.groupby(["env_step"])["std"]
        env_steps = grouped_data.mean().index.values
        mean_std = grouped_data.mean().values
        std_error = grouped_data.sem().values

        # Plot mean history with thicker lines
        if exp == 'SimbaV2':
            zorder = 10
        elif exp == 'Simba+':
            zorder = 0
        else:
            zorder = -10
            
        ax.plot(env_steps, 
                mean_std, 
                linewidth=line_width, 
                color=colors[exp], 
                zorder=zorder)
        
        # Draw markers
        timestep_list = [0, 2e5, 4e5, 6e5, 8e5, 1e6]
        markers_idx = list(range(0, len(mean_std), len(mean_std) // (num_x_ticks-1)))
        timestep_list = [t for t in timestep_list if t <= max(env_steps)]
        markers_idx = markers_idx[:len(timestep_list)]
        markers_values = np.interp(timestep_list, env_steps, mean_std)
        ax.scatter(timestep_list,
                    markers_values,
                    label=None,
                    color=colors[exp], 
                    edgecolor='white',
                    marker=markers[exp], 
                    s=marker_sizes[exp],
                    zorder=zorder+5)
            
        ax.fill_between(
            env_steps,
            mean_std - 1.96 * std_error,
            mean_std + 1.96 * std_error,
            alpha=0.2,
            color=colors[exp])
    
    # Set labels and limits
    ax.set_xlabel(x_label, fontsize=TICK_FONTSIZE)

    # Set x-ticks and format them
    x_ticks = np.linspace(0, 1e6, num_x_ticks)
    ax.set_xticks(x_ticks)
    # Format x-tick labels
    x_tick_labels = []
    for tick in x_ticks:
        if tick == 0:
            x_tick_labels.append("0")
        else:
            x_tick_labels.append("{:.1f}".format(tick / 1e6))
    
    ax.set_xticklabels(x_tick_labels, fontsize=TICK_FONTSIZE)
    
    # Set y-ticks
    y_ticks = np.linspace(y_lim_min, y_lim_max, num_y_ticks)
    ax.set_yticks(y_ticks)
    y_len = y_lim_max - y_lim_min
    ax.set_ylim(y_lim_min - 0.1*y_len, y_lim_max + 0.1*y_len)
    
    ax.set_autoscale_on(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(num_x_ticks, integer=True))
    ax.yaxis.set_major_locator(plt.MaxNLocator(num_y_ticks, integer=True))
    ax.xaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
    ax.yaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
    simple_axis(ax)
    
    import matplotlib.ticker as mticker

    formatter = mticker.ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((0, 1))
    ax.yaxis.set_major_formatter(formatter)
    ax.yaxis.get_offset_text().set_fontsize(TICK_FONTSIZE)
    ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)

    
    # Add custom title with first line bold
    title = "Mean Q Std"
    ax.set_title('')  # Clear any existing title
    ax.text(0.5, 1.08, title, color='black', transform=ax.transAxes, ha='center', va='bottom', fontsize=TITLE_FONTSIZE,)

In [186]:
def subplot_effective_lr_total(
    ax,
    experiments,
    metrics,
    prefixes,
    metric_data,
    colors,
    num_x_ticks: int = 6,
    num_y_ticks: int = 5,
    x_label: str = "Env steps (M)",
    line_width: float = 2.5,
    y_lim_mins: dict = {"encoder": 0, "predictor": 0},
    y_lim_maxs: dict = {"encoder": 1e-1, "predictor": 1.0},
    simbav2_lr: float = 1e-3
):  
    ax.yaxis.set_major_locator(plt.MaxNLocator(num_y_ticks, integer=True))
    ax.yaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
    for i, _metric in enumerate(metrics):
        prefix = prefixes[i]
        y_lim_min, y_lim_max = y_lim_mins[prefix], y_lim_maxs[prefix]
        _metric_data = metric_data[metric_data["metric"] == _metric]
        
        for exp in experiments:
            exp_data = _metric_data[_metric_data["exp_name"] == exp]
            if len(exp_data) == 0:
                continue
            
            # Group by env_step and calculate mean across all environments
            grouped_data = exp_data.groupby(["env_step"])["value"]
            env_steps = grouped_data.mean().index.values
            mean = grouped_data.mean().values
            std_error = grouped_data.sem().values  # Standard error of the mean

            # Plot mean history with thicker lines
            if exp == 'SimbaV2_encoder':
                zorder = 20
            elif exp == 'SimbaV2_predictor':
                zorder = 20
            elif exp == 'Simba_encoder':
                zorder = 0
            else:
                zorder = -10
            
            
            if exp == "Simba":
                color = "black"
                if i >= 1:
                    color = colors[f"{exp}_{prefix}_tick"]
                    ax = ax.twinx()
                    
                # Set ylimb
                if color:
                    ax.tick_params(axis='y', labelcolor=color, labelsize=TICK_FONTSIZE)
                else:
                    ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)
                y_len = y_lim_max - y_lim_min
                ax.set_ylim(-0.1 * y_len, y_lim_max + 0.1 * y_len)
                y_ticks = np.linspace(0, y_lim_max, num_y_ticks)
                
                y_ticks[0] = simbav2_lr
                ax.set_yticks(y_ticks)
                
                import matplotlib.ticker as mticker

                formatter = mticker.ScalarFormatter(useMathText=True)
                # formatter.set_powerlimits((0, 1))
                ax.yaxis.set_major_formatter(formatter)
                ax.yaxis.get_offset_text().set_fontsize(TICK_FONTSIZE)
        
            exp = f"{exp}_{prefix}"
            
            ax.plot(env_steps, 
                    mean, 
                    linewidth=line_width, 
                    color=colors[exp], 
                    zorder=zorder)
            
            # Draw markers
            timestep_list = [0, 2e5, 4e5, 6e5, 8e5, 1e6]
            markers_idx = list(range(0, len(mean), len(mean) // (num_x_ticks-1)))
            timestep_list = [t for t in timestep_list if t <= max(env_steps)]
            markers_idx = markers_idx[:len(timestep_list)]
            markers_values = np.interp(timestep_list, env_steps, mean)
            ax.scatter(timestep_list,
                        markers_values,
                        label=None,
                        color=colors[exp], 
                        edgecolor='white',
                        marker=markers[exp], 
                        s=marker_sizes[exp],
                        zorder=zorder+5)
                
            ax.fill_between(
                env_steps,
                mean - 1.96 * std_error,
                mean + 1.96 * std_error,
                alpha=0.2,
                color=colors[exp])

        # Set labels and limits
        ax.set_xlabel(x_label, fontsize=TICK_FONTSIZE)
        
        # Set x-ticks and format them
        x_ticks = np.linspace(0, 1e6, num_x_ticks)
        ax.set_xticks(x_ticks)
        # Format x-tick labels
        x_tick_labels = []
        for tick in x_ticks:
            if tick == 0:
                x_tick_labels.append("0")
            else:
                x_tick_labels.append("{:.1f}".format(tick / 1e6))
        
        ax.set_xticklabels(x_tick_labels, fontsize=TICK_FONTSIZE)
        
        ax.set_autoscale_on(False)
        ax.xaxis.set_major_locator(plt.MaxNLocator(num_x_ticks, integer=True))
        ax.xaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
        simple_axis(ax)
    
    # Add custom title with first line bold
    title = "Effective Learning Rate"
    ax.set_title('')  # Clear any existing title
    ax.text(0.5, 1.08, title, color="black", transform=ax.transAxes, ha='center', va='bottom', fontsize=TITLE_FONTSIZE,)

In [187]:
def plot_analysis_metrics_total(
    eval_df,
    env_names: list,
    colors,
    metrics: list,
    metrics_to_title: list,
    bbox_to_anchor_upper: tuple,
    bbox_to_anchor_bottom: tuple,
    num_x_ticks: int = 6,
    num_y_ticks: int = 5,
    x_label: str = "Env steps (M)",
    plot_width: int = 20,
    plot_height: int = 8,
    wspace: float = 0.2,
    hspace: float = 0.5,
    line_width: float = 2.5,
    y_lim_mins: list = [0, 0, None, None, 1, 0, 0, None],
    y_lim_maxs: list = [1, 1, None, None, 10, 120, 8, None],
    main_title: str = "DMC-Hard",
    columns: list = ["SimbaV2", "Simba"],
):  
    env_data = eval_df[eval_df["env_name"].isin(env_names)]
    experiments = env_data["exp_name"].unique()
    assert sorted(columns) == sorted(experiments)
    num_plots = 8
    ncols = 4
    nrows = (num_plots + ncols - 1) // ncols  # Calculate number of rows needed
    
    # Set up the figure
    sns.set_style('darkgrid', {"axes.facecolor": "white", "grid.color": BORDER_GRAY})
    # sns.set_style('whitegrid')    
    
    fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(plot_width, plot_height), squeeze=False)
    fig.subplots_adjust(hspace=hspace, wspace=wspace)
    axs = axs.flatten()  # Flatten in case of a single subplot
    
    # Sublabel
    axs[0].text(-0.3, 1.18, "(a)", ha="left", va="top", fontsize=LABEL_FONTSIZE, transform=axs[0].transAxes)
    axs[4].text(-0.3, 1.18, "(b)", ha="left", va="top", fontsize=LABEL_FONTSIZE, transform=axs[4].transAxes)
    
    # Start plotting
    for i, metric in enumerate(metrics):
        ax = axs[i]
        y_lim_min, y_lim_max = y_lim_mins[i], y_lim_maxs[i]
        
        # For upper subplots
        if i <= 3:
            if metric == "q_bias":
                metric_data = env_data[env_data["metric"].isin(['avg_discounted_return', 'avg_estimate_q'])]
                subplot_q_bias_total(
                    ax,
                    experiments,
                    metric_data,
                    colors)
                continue
            elif metric == "q_variance":
                metric_data = env_data[env_data["metric"].isin(['avg_discounted_return', 'avg_estimate_q'])]
                subplot_q_variance_total(
                    ax,
                    experiments,
                    metric_data,
                    colors)
                continue
            else: 
                metric_data = env_data[env_data["metric"] == metric]
            
            for exp in columns:
                exp_data = metric_data[metric_data["exp_name"] == exp]
                if len(exp_data) == 0:
                    continue  
                
                # Check if env_step = 0 exists, if not, add it with value 0
                if 0 not in exp_data["env_step"].values:
                    if metric in ["avg_return", "avg_discounted_return", "avg_estimate_q"]: 
                        default_value = 0
                        zero_row = pd.DataFrame({
                            "env_step": [0],
                            "value": [default_value],
                            "exp_name": [exp],
                            "metric": [metric]
                        })
                        exp_data = pd.concat([zero_row, exp_data]).reset_index(drop=True)
                    
                # Group by env_step and calculate mean across all environments
                grouped_data = exp_data.groupby(["env_step"])["value"]
                env_steps = grouped_data.mean().index.values
                mean = grouped_data.mean().values
                std_error = grouped_data.sem().values  # Standard error of the mean

                # Plot mean history with thicker lines
                if exp == 'SimbaV2':
                    zorder = 20
                else:
                    zorder = 10

                ax.plot(env_steps, 
                        mean, 
                        label=exp, 
                        linewidth=line_width, 
                        color=colors[exp], 
                        zorder=zorder)
                
                # Draw markers
                timestep_list = [0, 2e5, 4e5, 6e5, 8e5, 1e6]
                markers_idx = list(range(0, len(mean), len(mean) // (num_x_ticks-1)))
                timestep_list = [t for t in timestep_list if t <= max(env_steps)]
                markers_idx = markers_idx[:len(timestep_list)]
                markers_values = np.interp(timestep_list, env_steps, mean)
                ax.scatter(timestep_list,
                            markers_values,
                            label=None,
                            color=colors[exp], 
                            edgecolor='white',
                            marker=markers[exp], 
                            s=marker_sizes[exp],
                            zorder=zorder+5)
                
                # Fill between mean - std_error and mean + std_error
                ax.fill_between(
                    env_steps,
                    mean - 1.96 * std_error,
                    mean + 1.96 * std_error,
                    alpha=0.2,
                    color=colors[exp],
                )
                ax.xaxis.set_major_locator(plt.MaxNLocator(num_x_ticks, integer=True))
                ax.yaxis.set_major_locator(plt.MaxNLocator(num_y_ticks, integer=True))
                ax.xaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
                ax.yaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
                simple_axis(ax)
        
        # For bottom subplots
        else:
            if metric == "effective_lr_total":
                metric_data = env_data[env_data["metric"].isin(['critic_encoder/' + metric, 'critic_predictor/' + metric])]
                metrics = ['critic_encoder/' + metric, 'critic_predictor/' + metric]
                prefixes = ['encoder', 'predictor']
                subplot_effective_lr_total(
                    ax,
                    experiments,
                    metrics,
                    prefixes,
                    metric_data,
                    colors)
                continue
            else: 
                if 'total' in metric:
                    _metrics = ['critic_encoder/' + metric, 'critic_predictor/' + metric]
                    _exp_prefixes = ['encoder', 'predictor']
                else: 
                    _metrics = [metric]
                    _exp_prefixes = [None]
                    
            for j, _metric in enumerate(_metrics):
                metric_data = env_data[env_data["metric"] == _metric]
                _exp_prefix = _exp_prefixes[j]
                
                for exp in columns:
                    exp_data = metric_data[metric_data["exp_name"] == exp]
                    if len(exp_data) == 0:
                        continue  
                    
                    # Check if env_step = 0 exists, if not, add it with value 0
                    if 0 not in exp_data["env_step"].values:
                        if metric in ["avg_return", "avg_discounted_return", "avg_estimate_q"]: 
                            default_value = 0
                            zero_row = pd.DataFrame({
                                "env_step": [0],
                                "value": [default_value],
                                "exp_name": [exp],
                                "metric": [metric]
                            })
                            exp_data = pd.concat([zero_row, exp_data]).reset_index(drop=True)
                        
                        
                    # Group by env_step and calculate mean across all environments
                    grouped_data = exp_data.groupby(["env_step"])["value"]
                    env_steps = grouped_data.mean().index.values
                    mean = grouped_data.mean().values
                    std_error = grouped_data.sem().values  # Standard error of the mean

                    # Plot mean history with thicker lines
                    if exp == 'SimbaV2':
                        zorder = 20
                    else:
                        zorder = 10
                
                    if _exp_prefix:
                        plot_label = f"{exp} {_exp_prefix.capitalize()}"
                        exp = f"{exp}_{_exp_prefix}"
                        
                    ax.plot(env_steps, 
                            mean, 
                            label=plot_label, 
                            linewidth=line_width, 
                            color=colors[exp], 
                            zorder=zorder)
                    
                    # Draw markers
                    num_x_ticks = 6
                    timestep_list = [2e5, 4e5, 6e5, 8e5, 1e6]
                        # timestep_list = [1e5, 2.5e5, 4e5, 5.5e5, 7e5, 8.5e5, 1e6]
                    markers_idx = list(range(0, len(mean), len(mean) // (num_x_ticks-1)))
                    timestep_list = [t for t in timestep_list if t <= max(env_steps)]
                    markers_idx = markers_idx[:len(timestep_list)]
                    markers_values = np.interp(timestep_list, env_steps, mean)
                    ax.scatter(timestep_list,
                                markers_values,
                                label=None,
                                color=colors[exp], 
                                edgecolor='white',
                                marker=markers[exp], 
                                s=marker_sizes[exp],
                                zorder=zorder+5)
                    
                    # Fill between mean - std_error and mean + std_error
                    ax.fill_between(
                        env_steps,
                        mean - 1.96 * std_error,
                        mean + 1.96 * std_error,
                        alpha=0.2,
                        color=colors[exp],
                    )
                    ax.xaxis.set_major_locator(plt.MaxNLocator(num_x_ticks, integer=True))
                    ax.yaxis.set_major_locator(plt.MaxNLocator(num_y_ticks, integer=True))
                    ax.xaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
                    ax.yaxis.grid(True, linestyle='--', linewidth=1.0, alpha=1.0)
                    simple_axis(ax)
        
                
        # Set labels and limits
        ax.set_xlabel(x_label, fontsize=TICK_FONTSIZE)

        # Set x-ticks and format them
        x_ticks = np.linspace(0, 1e6, num_x_ticks)
            
        ax.set_xticks(x_ticks)
        # Format x-tick labels
        x_tick_labels = []
        for tick in x_ticks:
            if tick == 0:
                x_tick_labels.append("0")
            else:
                x_tick_labels.append("{:.1f}".format(tick / 1e6))
        
        ax.set_xticklabels(x_tick_labels, fontsize=TICK_FONTSIZE)
        ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)
        
        if "fnorm" in metric:
            y_len = y_lim_max - y_lim_min
            ax.set_ylim(y_lim_min - 0.1*y_len, y_lim_max + 0.1*y_len)
            y_ticks = np.linspace(0, y_lim_max, num_y_ticks)
            y_ticks[0] = 1
            ax.set_yticks(y_ticks)
        elif "gnorm" in metric:
            y_len = y_lim_max - y_lim_min
            ax.set_ylim(y_lim_min - 0.1*y_len, y_lim_max + 0.1*y_len)
            y_ticks = np.linspace(0, y_lim_max, num_y_ticks)
            y_ticks[0] = 1e-4
            ax.set_yticks(y_ticks)
            import matplotlib as mpl
            mpl.rcParams.update(mpl.rcParamsDefault)
            from matplotlib.ticker import FuncFormatter
            def scientific_formatter(val, pos):
                """Formats the tick value in scientific notation with e and removes leading zero in exponent."""
                formatted = f"{val:.1e}"  # Format in scientific notation
                base, exp = formatted.split("e")
                if float(base).is_integer():
                    base = int(float(base))
                    if not base == 1:
                        return base
                output = "{" + f"{int(exp)}" + "}"
                return f"$10^{output}$"

            ax.yaxis.set_major_formatter(FuncFormatter(scientific_formatter))
            for label in ax.yaxis.get_majorticklabels():
                if label.get_text() == r"$10^{-4}$":  # Identify the 10^{-4} label
                    label.set_x(-0.01)  # Adjust its horizontal position slightly
                    label.set_horizontalalignment('center')  # Center-align it
            
        else:
            y_ticks = np.linspace(y_lim_min, y_lim_max, num_y_ticks)
            ax.set_yticks(y_ticks)
            y_len = y_lim_max - y_lim_min
            ax.set_ylim(y_lim_min - 0.1*y_len, y_lim_max + 0.1*y_len)
        
        # Add custom title with first line bold
        title = metrics_to_title[metric]
        ax.set_title('')  # Clear any existing title
        ax.text(0.5, 1.08, title, color='black', transform=ax.transAxes, ha='center', va='bottom', fontsize=TITLE_FONTSIZE,)
        
        ax.set_autoscale_on(False)

    # Add shared legend below the subplots
    _, labels = axs[0].get_legend_handles_labels()
    plt.subplots_adjust(bottom=0.15)

    # Add markers to the legend
    handles = [legend_handles[k] for k in labels]
    leg = fig.legend(handles, 
            labels, 
            loc='lower center', 
            ncol=len(labels), 
            bbox_to_anchor=bbox_to_anchor_upper, 
            fontsize='x-large',
            handletextpad=0.1,
            )
    leg.get_frame().set_linewidth(0.0)
    
    # Add shared legend below the subplots
    _, labels = axs[7].get_legend_handles_labels()
    plt.subplots_adjust(bottom=0.15)

    # Add markers to the legend
    labels = ["SimbaV2 Encoder", "SimbaV2 Predictor", "Simba Encoder", "Simba Predictor"]
    handles = [legend_handles[k] for k in labels]
    leg = fig.legend(handles, 
            labels, 
            loc='lower center', 
            ncol=len(labels), 
            bbox_to_anchor=bbox_to_anchor_bottom, 
            fontsize='x-large',
            handletextpad=0.1,
            )
    leg.get_frame().set_linewidth(0.0)
    

    # Adjust layout and display
    # plt.suptitle(f"{main_title}")
    plt.savefig(f"{main_title}_analysis.png", bbox_inches='tight')
    plt.savefig(f"{main_title}_analysis.pdf", bbox_inches='tight')
    plt.close()

In [188]:
metrics_to_title = {
    'avg_return': "Average Return",
    'avg_estimate_q': "Average Q Prediction",
    'q_bias': "Norm Q Bias",
    'q_variance': "Mean Q Std",
    'effective_fnorm_total': "Feature Norm",
    'effective_pnorm_total': "Parameter Norm",
    'effective_gnorm_total': "Gradient Norm",
    'effective_lr_total': "Effective Learning Rate",
}

In [189]:
plot_analysis_metrics_total(
    eval_df,
    HB_LOCOMOTION_NOHAND,
    colors,
    metrics=['avg_return', 'avg_estimate_q', 'q_bias', 'q_variance', 'effective_fnorm_total', 'effective_pnorm_total', 'effective_gnorm_total', 'effective_lr_total'],
    metrics_to_title=metrics_to_title,
    hspace=0.75,
    bbox_to_anchor_upper=(0.5, 0.475),
    bbox_to_anchor_bottom=(0.5, 0.01),
    main_title="sec5",
    columns=["SimbaV2", "Simba"],
)